## Задание 8

Напишите функцию для подсчета количества параметров в сверточном слое. Ваша функция должна принимать на вход количество входных каналов, количество выходных каналов, размер ядра и стоит ли делать обучаемый сдвиг. Сделайте для нее такую сигнатуру: `def count_parameters_conv(in_channels: int, out_channels: int, kernel_size: int, bias: bool):`



In [ ]:
def count_parameters_conv(in_channels: int, out_channels: int, kernel_size: int, bias: bool):
    return out_channels * (in_channels * kernel_size * kernel_size + int(bias))

## Задание 9

Обучите полносвязную нейронную сеть для классификации на датасете **MNIST**. Добейтесь качества в 98% на тестовой выборке.

Можете использовать уже написанные ранее функции `train` и `evaluate`.

Чтобы мы могли проверить вашу модель, напишите функцию, которая создает вашу модель и возвращает объект, назовите функцию `create_mlp_model`, без аргументов. Вам понадобится сохранить веса вашей модели и сдать их в тестировщик, для этого воспользуйтесь методами `torch.save` и `state_dict`.



In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from IPython.display import clear_output
from torch.optim import Adam
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from tqdm import tqdm


def set_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)


def train(
    model: nn.Module, data_loader: DataLoader, optimizer: Optimizer, loss_fn
) -> tuple[float, float]:
    model.train()

    total_loss = 0
    total = 0
    correct = 0

    for x, y in tqdm(data_loader, desc='Train'):
        optimizer.zero_grad()

        output = model(x)

        loss = loss_fn(output, y)

        loss.backward()

        total_loss += loss.item()

        optimizer.step()

        _, y_pred = torch.max(output, 1)
        total += y.size(0)
        correct += (y_pred == y).sum().item()

    return total_loss / len(data_loader), correct / total


@torch.inference_mode()
def evaluate(model: nn.Module, data_loader: DataLoader, loss_fn) -> tuple[float, float]:
    model.eval()

    total_loss = 0
    total = 0
    correct = 0

    for x, y in tqdm(data_loader, desc='Evaluate'):
        output = model(x)

        loss = loss_fn(output, y)

        total_loss += loss.item()

        _, y_pred = torch.max(output, 1)
        total += y.size(0)
        correct += (y_pred == y).sum().item()

    return total_loss / len(data_loader), correct / total


def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    train_accuracy: list[float],
    valid_accuracy: list[float],
    title: str,
):
    plt.figure(figsize=(16, 8))

    plt.title(title + " loss")

    plt.plot(train_loss, label="Train loss")
    plt.plot(valid_loss, label="Valid loss")
    plt.legend()
    plt.grid()

    plt.show()

    plt.figure(figsize=(16, 8))

    plt.title(title + " accuracy")

    plt.plot(train_accuracy, label="Train accuracy")
    plt.plot(valid_accuracy, label="Valid accuracy")
    plt.legend()
    plt.grid()

    plt.show()


def whole_train_valid_cycle(
    model, train_loader, valid_loader, optimizer, loss_fn, threshold, title
):
    train_loss_history, valid_loss_history = [], []
    train_accuracy_history, valid_accuracy_history = [], []

    for epoch in range(100):
        train_loss, train_accuracy = train(model, train_loader, optimizer, loss_fn)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        train_accuracy_history.append(train_accuracy)
        valid_accuracy_history.append(valid_accuracy)

        clear_output(wait=True)

        plot_stats(
            train_loss_history,
            valid_loss_history,
            train_accuracy_history,
            valid_accuracy_history,
            title,
        )

        if valid_accuracy >= threshold:
            break


def main(func, threshold, title):
    set_seed(0xDEADF00D)

    mnist_train = MNIST("mnist", train=True, download=True, transform=T.ToTensor())

    mnist_valid = MNIST("mnist", train=False, download=True, transform=T.ToTensor())

    train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
    valid_loader = DataLoader(mnist_valid, batch_size=64, shuffle=False)

    model = func()

    optimizer = Adam(model.parameters(), lr=1e-3)

    loss_fn = nn.CrossEntropyLoss()

    whole_train_valid_cycle(
        model, train_loader, valid_loader, optimizer, loss_fn, threshold, title
    )

    torch.save(model.state_dict(), "model.pt")


In [ ]:
# файл для сдачи
import torch.nn as nn

def create_mlp_model() -> nn.Module:
    return nn.Sequential(
        nn.Flatten(), nn.Linear(28 * 28, 256), nn.ReLU(), nn.Linear(256, 10)
    )

In [ ]:
main(create_mlp_model, 0.98, "MLP")

## Задание 10

Обучите сверточную нейронную сеть для классификации на датасете **MNIST**. Добейтесь качества в 99.3% на тестовой выборке.

Можете использовать уже написанные ранее функции `train` и `evaluate`.

Чтобы мы могли проверить вашу модель, напишите функцию, которая создает вашу модель и возвращает объект, назовите функцию `create_conv_model`, без аргументов. Вам понадобится сохранить веса вашей модели и сдать их в тестировщик, для этого воспользуйтесь методами `torch.save` и `state_dict`.



In [ ]:
# файл для сдачи
import torch.nn as nn


def create_conv_model():
    return nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, padding=1),
        nn.ReLU(),

        nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),

        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),

        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=1),
        nn.ReLU(),

        nn.Flatten(),
        nn.Linear(128 * 7 * 7, 512),
        nn.LeakyReLU(),
        nn.Linear(512, 10)
    )

In [ ]:
main(create_conv_model, 0.993, "CONV")